# 🚀 Train YOLO11 on a Free Colab GPU (T4)

This notebook does the **same training as `scripts/train_yolo.py`** — but on Google Colab's free **T4 GPU**, so a full 100-epoch run takes **~20–40 minutes** instead of ~20+ hours on CPU.

**How to use (once):**
1. **Runtime ▸ Change runtime type ▸ T4 GPU** (free tier includes it)
2. Run the cells top to bottom
3. When asked, upload your dataset zip (a Roboflow export: `train/`, `valid/`, `test/`, `data.yaml`)
4. Download `best.pt` from the last cell and drop it into your SemarangVision project

The model file is **hardware-agnostic** — training on GPU here, then running on CPU in your backend, is fully supported.


In [ ]:
# 1. Check the GPU and install ultralytics
#    (Colab already ships CUDA-enabled PyTorch — no separate torch install needed)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics


## 2. Dataset

Two ways to get your dataset here:
- **Upload (easiest):** run the "Get the dataset" cell and pick your dataset `.zip` (e.g. a Roboflow export) in the file dialog.
- **Google Drive:** set `USE_DRIVE = True` in the next cell and put the zip somewhere in Drive.


In [ ]:
# 2. (Optional) Mount Google Drive — skip this if you'll upload the zip directly
USE_DRIVE = False  # set to True to pull the dataset from Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Drive mounted.")


In [ ]:
# 3. Configuration — edit these before training
DATASET_ZIP_IN_DRIVE = "/content/drive/MyDrive/flood-detection-6.zip"  # only used when USE_DRIVE=True
DATASET_ROOT = "/content/flood"   # where the dataset lives inside Colab

MODEL    = "yolo11s.pt"   # yolo11n.pt = fastest/lightest, yolo11s.pt = better accuracy
EPOCHS   = 100            # full-quality run; a T4 chews through this in ~20-40 min
IMGSZ    = 640            # YOLO default; 416/320 trains faster
BATCH    = 16             # -1 lets ultralytics auto-pick the best size
DEVICE   = 0              # 0 = the T4 GPU; 'cpu' also works (much slower)
PROJECT  = "/content/runs/detect"
NAME     = "flood_yolo11s"
WORKERS  = 2              # keep low — Colab's CPUs are shared
PATIENCE = 50             # auto-stop if validation stops improving
SEED     = 0


In [ ]:
# 4. Get the dataset into Colab (upload the zip, or pull it from Drive)
import os
import zipfile
from pathlib import Path
from google.colab import files

root = Path(DATASET_ROOT)
if not root.exists():
    zip_src = None
    if USE_DRIVE and os.path.exists(DATASET_ZIP_IN_DRIVE):
        zip_src = DATASET_ZIP_IN_DRIVE
        print(f"Using dataset from Drive: {zip_src}")
    else:
        print("Upload your dataset .zip (Roboflow export: train/, valid/, test/, data.yaml)...")
        uploaded = files.upload()
        zip_src = next(iter(uploaded))
    with zipfile.ZipFile(zip_src) as zf:
        zf.extractall(root)
    print(f"Dataset extracted to {root}: {sorted(p.name for p in root.iterdir())}")
else:
    print(f"Dataset already present at {root}: {sorted(p.name for p in root.iterdir())}")

# Some Roboflow exports wrap everything in a single top-level folder — unwrap it
if not (root / "data.yaml").is_file() and not (root / "train").is_dir():
    inner = [p for p in root.iterdir() if p.is_dir()]
    if len(inner) == 1:
        for item in inner[0].iterdir():
            item.rename(root / item.name)
        inner[0].rmdir()
        print(f"Unwrapped nested export folder — now: {sorted(p.name for p in root.iterdir())}")


In [ ]:
# 5. Build a robust data.yaml (absolute paths) — same logic as scripts/train_yolo.py
import yaml
from pathlib import Path

_SPLIT_LAYOUTS = {
    "train": [("train/images", "train/labels"), ("images/train", "labels/train")],
    "val": [
        ("val/images", "val/labels"),
        ("valid/images", "valid/labels"),
        ("images/val", "labels/val"),
        ("images/valid", "labels/valid"),
    ],
}


def detect_split(data_root, split):
    for images_rel, labels_rel in _SPLIT_LAYOUTS[split]:
        if (data_root / images_rel).is_dir():
            return images_rel, labels_rel
    return None


def resolve_dataset(data_root):
    data_root = Path(data_root).resolve()
    if not data_root.is_dir():
        raise FileNotFoundError(f"Dataset folder not found: {data_root}")

    # Read class names from an existing data.yaml if there is one
    names = None
    existing = data_root / "data.yaml"
    if existing.is_file():
        cfg = yaml.safe_load(existing.read_text(encoding="utf-8")) or {}
        names = cfg.get("names")
        if isinstance(names, dict):             # {0: 'Banjir', ...}
            names = [names[k] for k in sorted(names, key=int)]
        elif isinstance(names, (list, tuple)):  # ['Banjir']
            names = list(names)

    train = detect_split(data_root, "train")
    val = detect_split(data_root, "val")
    if train is None or val is None:
        raise RuntimeError(
            f"Could not find a train/val split in {data_root}. "
            f"Contents: {sorted(p.name for p in data_root.iterdir())}"
        )

    if names is None:  # derive the class count from the label files
        max_id = -1
        for _, labels_rel in (train, val):
            for label_file in (data_root / labels_rel).glob("*.txt"):
                for line in label_file.read_text(encoding="utf-8").splitlines():
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        max_id = max(max_id, int(line.split()[0]))
                    except ValueError:
                        raise ValueError(
                            f"Malformed label line in {label_file}: {line!r}"
                        )
        names = [f"class_{i}" for i in range(max_id + 1)]

    out = data_root / "data_colab.yaml"
    out.write_text(
        f"path: {data_root.as_posix()}\n"
        f"train: {train[0]}\n"
        f"val: {val[0]}\n"
        f"nc: {len(names)}\n"
        f"names: {names}\n",
        encoding="utf-8",
    )
    print("Using data config:\n" + out.read_text(encoding="utf-8"))
    return out


DATA_YAML = resolve_dataset(DATASET_ROOT)


## 3. Train

Training progress bars, loss curves and validation metrics stream live here. Every epoch saves `best.pt` / `last.pt`; training auto-stops early if validation stops improving (`PATIENCE`).

> **Colab disconnected mid-run?** Just re-run the training cell with `resume=True` added to `model.train(...)` — it continues from `last.pt` instead of starting over.

In [ ]:
# 6. Train on the GPU
import time

import torch
from ultralytics import YOLO

if DEVICE != "cpu" and not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA GPU detected. Go to Runtime > Change runtime type > T4 GPU, "
        "or set DEVICE = 'cpu'."
    )

model = YOLO(MODEL)
t0 = time.time()
model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=DEVICE,
    project=PROJECT, name=NAME, workers=WORKERS, patience=PATIENCE, seed=SEED,
)
minutes = (time.time() - t0) / 60
print(f"\nDone in {minutes:.1f} min — best weights: {model.trainer.best}")
print(f"Run directory: {model.trainer.save_dir}")


In [ ]:
# 7. Summary of the final metrics + learning curves
import csv
from pathlib import Path

from IPython.display import Image, display

save_dir = Path(model.trainer.save_dir)
csv_path = save_dir / "results.csv"
if not csv_path.is_file():
    raise RuntimeError("No results.csv yet — complete (or resume) training first.")
rows = list(csv.DictReader(csv_path.read_text().splitlines()))
best = max(rows, key=lambda r: float(r["metrics/mAP50(B)"]))
last = rows[-1]
print(f"Best epoch: {best['epoch']}   (mAP50 = {float(best['metrics/mAP50(B)']):.3f})")
for k in ("metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"):
    print(f"  final {k.split('/')[-1]:<10}: {float(last[k]):.4f}")

img = save_dir / "results.png"
if img.exists():
    display(Image(filename=str(img)))


In [ ]:
# 8. Download best.pt (+ plots + results.csv) to your machine
import zipfile
from google.colab import files

weights = Path(model.trainer.best)
print(f"Model file: {weights} ({weights.stat().st_size / 1e6:.1f} MB)")

archive = "/content/flood_yolo11s_run.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(weights, "best.pt")
    for extra in sorted(save_dir.glob("*.png")) + [save_dir / "results.csv"]:
        if extra.exists():
            zf.write(extra, extra.name)
files.download(archive)
print("Download started — save best.pt into your project, then use the local snippet below.")


In [ ]:
# 9. (Optional) Sanity check on the test split
test_dir = Path(DATASET_ROOT) / "test" / "images"
if not test_dir.is_dir():
    test_dir = Path(DATASET_ROOT) / "images" / "test"
if test_dir.is_dir():
    res = model.predict(str(test_dir), imgsz=IMGSZ, device=DEVICE, conf=0.25, save=True)
    print("Predictions saved under", Path(res[0].save_dir))
else:
    print("No test split found — skipping.")


## Back on your machine (CPU inference)

The downloaded model is the **same weights file** — training on GPU changes nothing for deployment. Copy it into the project and run it on CPU:

```bash
# from your SemarangVision-BE project folder
mkdir -p models
# move the best.pt you downloaded (your browser's Downloads folder) into the project
mv ~/Downloads/best.pt models/

# sanity check on the test split (CPU — a few seconds per image)
uv run python -c "from ultralytics import YOLO; m = YOLO('models/best.pt'); m.predict('datasets/flood/test/images', imgsz=640, device='cpu', save=True)"
```

No extra dependencies — `ultralytics` is already in `pyproject.toml`. The detection pipeline later just loads `best.pt` and calls `model.predict(frame, device='cpu')`.
